# Lab 6.3 &mdash; ChromaDB Basics

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Create a Chroma collection and add documents that carry metadata
- Decide what metadata to attach &mdash; before you need it, because you cannot filter on what you did not store
- Query by meaning and read the <code>distances</code> nobody reads
- Scope a search with a <code>where</code> filter, which is the half of retrieval that is exact

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **Raw `chromadb` here, not the LangChain wrapper.** One level down is worth seeing
> once; Lab 6.5 puts the wrapper back on top.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Ten short passages from a company handbook. Note what each one carries besides its text:
# a category, a source file and a page. Those three are what Lab 6.3 filters on and what
# Lab 6.7 cites -- metadata is not decoration, it is the half of retrieval that is exact.
#
# Note also what is NOT here: nothing mentions salary, notice period or the share price.
# Labs 6.6 and 6.8 need that gap, because refusing is a feature.

HANDBOOK = [
    {"text": "Annual leave is 24 days per year for full-time employees. Leave must be applied "
             "for at least 3 working days in advance. Unused annual leave cannot be carried "
             "forward to the next financial year.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Sick leave is 12 days per year. Notify your manager by 10 AM on the day of "
             "absence. A medical certificate is required for absences of more than 2 "
             "consecutive days.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks. Both "
             "must be applied for at least 30 days before the expected date.",
     "category": "leave", "source": "handbook.pdf", "page": 6},
    {"text": "Employees may work from home up to 3 days per week with team lead approval. "
             "Core hours are 10 AM to 4 PM IST, and you must be reachable during them.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "A VPN connection is mandatory for reaching internal systems from home. "
             "Contact the IT helpdesk for VPN setup.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "Internet reimbursement is 1,500 per month for employees working from home. "
             "Submit the broadband bill to finance by the 5th of each month.",
     "category": "expense", "source": "handbook.pdf", "page": 9},
    {"text": "Travel expenses must be submitted with original receipts within 7 working days "
             "of travel. The meal allowance during client visits is 500 per day.",
     "category": "expense", "source": "handbook.pdf", "page": 12},
    {"text": "Laptops are provided by the company and replaced every 3 years. Software "
             "licence requests go through the IT helpdesk and must not be bought directly.",
     "category": "tech", "source": "tech-guide.pdf", "page": 7},
    {"text": "The backend stack is Python with FastAPI, and Java with Spring Boot. New "
             "services should use Python unless there is a specific reason not to. "
             "PostgreSQL is the primary database.",
     "category": "tech", "source": "tech-guide.pdf", "page": 3},
    {"text": "The Bangalore office is the headquarters, on the 5th floor, with 200+ staff. "
             "The Mumbai office is in the Worli business district, Tower A, 12th floor.",
     "category": "office", "source": "office-directory.pdf", "page": 15},
]

print(f"{len(HANDBOOK)} passages, "
      f"{len({d['category'] for d in HANDBOOK})} categories, "
      f"{len({d['source'] for d in HANDBOOK})} source files")

## Concept

A **collection** is the unit of separation &mdash; one per corpus, or per tenant. Inside it,
every document is four things:

| | |
|---|---|
| `id` | how you update or delete it later. People forget it exists until re-indexing day |
| `embedding` | the vector it is ranked by |
| `document` | the original text |
| `metadata` | anything else: source, page, category, entitlement |

Two of those you choose. The embedding is computed for you &mdash; here by the same
`all-MiniLM-L6-v2` you used in Lab 6.2, handed to Chroma as an *embedding function* so it
does not go looking for a model of its own.

## Section 1 &mdash; A collection, and the metadata you will wish you had stored

You cannot filter on an attribute you did not attach. Deciding the metadata schema is the
one irreversible decision in this lab: changing it later means re-embedding the corpus.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# The same all-MiniLM-L6-v2 as Lab 6.2 -- here it is simply Chroma's own default, which
# is why raw chromadb needs no embedding configuration at all.
EMBED_FN = DefaultEmbeddingFunction()

def metadata_for(passage: dict) -> dict:
    """What has to travel with the text so a later query can scope to it?

    Lab 6.7 needs to cite the answer, and this lab needs to filter by topic.
    Both of those are only possible if the value is in here."""
    return {"category": passage["category"], "source": passage["source"],
            "page": passage["page"]}


def build_collection():
    """Given -- one add() call. Note that ids are ours to choose, not Chroma's."""
    client = chromadb.Client()                       # in-memory: fast, nothing to clean up
    col = client.get_or_create_collection("handbook", embedding_function=EMBED_FN)
    if col.count() == 0:
        col.add(documents=[d["text"] for d in HANDBOOK],
                metadatas=[metadata_for(d) for d in HANDBOOK],
                ids=[f"doc{i}" for i in range(len(HANDBOOK))])
    return col

In [ ]:
# --- Self-check: Section 1   (a real Chroma collection -- still no gateway)
check("every passage is in the collection",
      lambda: build_collection().count() == len(HANDBOOK))
check("metadata carries the source file",
      lambda: all("source" in m for m in build_collection().get()["metadatas"]),
      "without it, Lab 6.7 has nothing to cite")
check("metadata carries the page",
      lambda: all(isinstance(m.get("page"), int)
                  for m in build_collection().get()["metadatas"]))
check("and the category, which is what the filter below needs",
      lambda: {m["category"] for m in build_collection().get()["metadatas"]}
              == {"leave", "wfh", "expense", "tech", "office"})

## Section 2 &mdash; Asking, and scoping

Two things to notice in this section, and the second one is the important one.

First, the query and the document share no words and match anyway. Second, `n_results=3`
returns three results whatever is in the corpus &mdash; ask about the share price and you
still get three handbook passages back, each with a distance that is the only clue anything
is wrong.

In [ ]:
def search(col, question: str, k: int = 3, where: dict | None = None) -> list:
    """Given. Returns (distance, text, metadata) triples, closest first."""
    res = col.query(query_texts=[question], n_results=k,
                    **({"where": where} if where else {}))
    return list(zip(res["distances"][0], res["documents"][0], res["metadatas"][0]))


def expense_only() -> dict:
    """A `where` clause that keeps the search inside expense passages.

    Chroma takes a dict of {field: value} against the metadata you stored above."""
    return {"category": "expense"}


def from_the_tech_guide() -> dict:
    """And one that keeps it inside a single source FILE, whatever the topic."""
    return {"source": "tech-guide.pdf"}

In [ ]:
# --- Self-check: Section 2
def _col():
    return build_collection()

check("a SPECIFIC question finds the annual-leave passage",
      lambda: "24 days" in search(_col(), "How much annual leave do I get?")[0][1],
      "neither 'much' nor 'get' appears in it -- the match is on meaning, not words")
check("a VAGUE question does not",
      lambda: "24 days" not in search(_col(), "How many holidays do I get?")[0][1],
      "'holidays' fits 'Sick leave is 12 days per year' just as well, and that one wins")
check("...and it loses by a rounding error, so top-1 was a coin flip",
      lambda: abs(search(_col(), "How many holidays do I get?", k=2)[1][0]
                  - search(_col(), "How many holidays do I get?", k=2)[0][0]) < 0.05,
      "two passages 0.001 apart -- this is the argument for k > 1, made by the data")
check("the expense filter returns expense passages and nothing else",
      lambda: all(m["category"] == "expense"
                  for _, _, m in search(_col(), "What can I claim?", where=expense_only())))
check("the source filter returns only the tech guide",
      lambda: all(m["source"] == "tech-guide.pdf"
                  for _, _, m in search(_col(), "What do we use?", k=2,
                                        where=from_the_tech_guide())))
check("an off-topic question STILL returns k results",
      lambda: len(search(_col(), "What is the company share price?", k=3)) == 3,
      "nothing refuses -- the only signal that this went wrong is the distance")
check("...and they are further away than a real hit",
      lambda: search(_col(), "What is the company share price?")[0][0]
              > search(_col(), "How many holidays do I get?")[0][0])

def _show():
    for label, q, w in [("specific  ", "How much annual leave do I get?", None),
                        ("vague     ", "How many holidays do I get?", None),
                        ("off topic ", "What is the company share price?", None),
                        ("filtered  ", "What can I claim?", expense_only())]:
        for d, t, m in search(_col(), q, k=2, where=w)[:2]:
            print(f"  {label} [{d:.4f}] ({m['source']} p.{m['page']}) {t[:52]}...")
        label = "          "
guard(_show)

In [ ]:
score()

## Your turn

1. Print the distance of the best hit for ten questions, five of which the handbook cannot
   answer. Where would you put a threshold? Now notice that the two groups overlap, and that
   any threshold you pick costs you something in one direction or the other.
2. Chroma also takes `{"$or": [...]}`. Write a filter for leave **or** wfh and ask
   &ldquo;what are my benefits?&rdquo;. Compare with no filter at all.
3. Add a `sensitivity` field to `metadata_for` and set it to `"restricted"` on the office
   passages. Now write the filter that a search would need if the person asking is not
   allowed to see restricted material. That is entitlement-aware retrieval, and it is the
   same `where` clause.